# iCloud Passwords to Nextcloud Passwords

This program reads the CSV file generated by [macOS Passwords app](https://support.apple.com/guide/passwords/create-a-password-mchla6acb8cb/mac), containing all your iCloud-stored passwords, and converts them to [Nextcloud Passwords](https://apps.nextcloud.com/apps/passwords) (NCP) database format, ready to be imported by this Nextcloud extension.

Screenshot of Apple Passwords app:

![Apple Passwords app in macOS](https://help.apple.com/assets/6901602838BFD0533000F123/69016029DE4C822607079200/en_US/6a040c5c07acc5be860c6cc66ac1a2f3.png)

Screenshot of Nextcloud Password app:

![Nextcloud Passwords app](https://raw.githubusercontent.com/wiki/marius-wieschollek/passwords/Users/_files/feature-overview.gif)

Apple Passwords app stores password entries a little different (and better) than other password apps:

- One entry may contain a list of sites where this credential should be used
- Stores the `otpauth://totp?...` URL used to create a one time code generator (a.k.a. OTP, a.k.a. Google Auth app)
- Stores passkeys (not exported in the CSV)
- Will mark as favorite (⭐) the entries whose label is in the `favorites` list below. Labels exist when you edit the title of the password entry in iOS or macOS Passwords app. Otherwise CSV entries will have labels generated from site address and username.

A credential entry with multiple sites will be exported to multiple CSV lines, making an annoyingly poluted large number of repeated entries. So this script will:

- Detect these repeated lines, aggregate them into one NCP entry with custom fileds for these extra sites
- Make custom field for the OTP URL
- Make deterministic UUIDs for each entry, so repeated imports into NCP won't create duplicates

Once the JSON file is generated, go to your Nextcloud URL https://${NEXCLOUD_ROOT}/<b>apps/passwords/#/backup/import</b> and select format "**Database Backup**" to import the JSON file.

![Nextcloud Passwords import screen](https://github.com/user-attachments/assets/dcfcd612-27cf-4554-af1a-ce2aca0fa9b7)

## Delete old passwords from NCP

Before importing a fresh file into NCP, you might want to delete previously created passwords.
Easiest way to do it is in the command line of your Nextcloud server.

Invoke Nextcloud's `occ passwords:user:delete` command passing your Nextcloud user:

```shell
NC_USER=$SUDO_USER
(cd /usr/share/nextcloud; echo yes | runuser -u apache -- ./occ passwords:user:delete $NC_USER )
```

This is what I do in my Nextcloud instance running on a Fedora server, installed from Fedora official packages.

I'm sure this could be acomplished too by using some Nextcloud API, but oh boy...

---

Passwords, OTPs, captchas are all obsolete. Sites and apps should be using Passkeys by now.

# Code to generate Nextcloud Passwords database import file

## Read file and restructure for NCP
Read iCloud exported passwords and transform structure into something compatible with Nextcloud Passwords

In [ ]:
favorites=[
    'My Linux user',
    'Microsoft Account',
    'Apple ID',
]

icloud_passwords='Passwords.csv'

In [ ]:
import os
import uuid
import json
import datetime
import hashlib
import numpy
import pandas

mtime=os.path.getmtime(icloud_passwords)

def uuid7(timestamp):
    """Create UUIDv7 from a Unix timestamp in milliseconds."""
    timestamp_ms = int(timestamp)

    # 48-bit timestamp + 76 random bits
    random_bits = uuid.uuid4().int & ((1 << 76) - 1)

    value = (timestamp_ms << 80) | random_bits

    # Set version = 7
    value &= ~(0xf << 76)
    value |= 7 << 76

    # Set variant = RFC 9562
    value &= ~(0b11 << 62)
    value |= 0b10 << 62

    return uuid.UUID(int=value)

folders = [
    dict(
        label       = f'iCloud Paswords as of {datetime.datetime.fromtimestamp(mtime):%Y-%m-%d %H:%M:%S}',

        id          = uuid7(mtime*1000).urn.split(':')[2],
        revision    = uuid7(mtime*1000).urn.split(':')[2],
        parent      = '00000000-0000-0000-0000-000000000000',
        edited      = int(mtime),
        favorite    = False
    ),
]

passwords = (
    pandas.read_csv(icloud_passwords)
    .reset_index(drop=False)
    .rename(
        columns=dict(
            Title='label',
            URL='url',
            Username='username',
            Password='password',
            Notes='notes',
            index='order',
        )
    )

    .assign(
        # Make a hash for future aggregations
        final_hash=lambda table: table.apply(
            axis=1,
            func=lambda row: (
                hashlib.shake_128(
                    '•'.join([
                        '' if pandas.isna(v) else str(v)
                        for v in [row.username,row.password,row.notes,row.OTPAuth]
                    ])
                    .encode()
                )
                .hexdigest(10)
            )
        ),

        # Keep order of items from original file using a sequence of timestamps from file time
        file_time=lambda table: pandas.Timestamp.fromtimestamp(mtime - 5000),
        edited=lambda table: (table.file_time +  pandas.to_timedelta(table.index, unit="s")).astype(int) // 10**9
    )

    # Aggregate by hash and define values for each other relevant field
    .groupby('final_hash')
    .agg(
        **dict(
            {
                f: (f,'first')
                for f in """
                    label
                    url
                    username
                    password
                    notes
                    edited
                    OTPAuth
                    order
                """.split()
            },
            URLs       = ('url', list)
        )
    )

    # Restructure some fields to be compatible with Nextcloud Passwords DB layout
    .assign(
        # Make a deterministic UUID based on label and username
        id=lambda table: table.apply(
            lambda row: uuid.uuid5(
                uuid.NAMESPACE_DNS,
                row.label or '' +
                row.username or ''
            ).urn.split(':')[2],
            axis=1
        ),

        revision=uuid7(mtime*1000).urn.split(':')[2],

        # Mark favorite entries based on list above
        favorite=lambda table: table.apply(lambda row: True if row.label in favorites else False, axis=1),

        # Prepare for custom fields
        custom=lambda table: table.apply(
            axis=1,
            func=lambda row:
                # custom fields for each extra URL found
                [dict(label='URLs',type='url',value=u) for u in row.URLs[1:] if len(row.URLs)>1 and pandas.notna(u)] +
                # custom field for OTPAuth URL
                [dict(label='OTPAuth',type='url',value=row.OTPAuth) if pandas.notna(row.OTPAuth) else None]
        ),

        # Delete None from final custom fields
        customFields=lambda table: table.custom.apply(lambda c: [i for i in c if i is not None] ),

        folder=folders[0]['id']
    )

    .fillna(
        dict(
            username='',
            password='',
            notes='',
            url='',
        )
    )

    ## Delete the auxiliary hash
    .reset_index(drop=True)

    ## Put in same order that the input
    .sort_values('order')

    # Export to Nextcloud Passwords format

    ## Only features that matter
    ['id label edited revision favorite username password url customFields notes folder'.split()]

    ## NCP format is simply a list of dicts
    .to_dict(orient='records')
)

## Envelope for NCP database file

In [ ]:
nextcloud_passwords_db = dict(
    version        = 3,
    encrypted      = False,
    tags           = [],
    folders        = folders,
    passwords      = passwords,
)

## Export to JSON

In [ ]:
with open(f"Nextcloud Passwords from iCloud [{datetime.datetime.fromtimestamp(mtime):%Y.%m.%d-%H.%M.%S}].json", "w") as f:
    json.dump(nextcloud_passwords_db, f, indent=4)

# Sample of generated file

```json
{
  "version": 3,
  "encrypted": false,
  "passwords": [
    {
      "id": "339a6628-2942-4346-939d-07376299e211",
      "revision": "1530fbcd-4d9c-4e4c-92d0-8c4c660d322e",
      "label": "Some site",
      "username": "usuer",
      "password": "password",
      "notes": "Some text\n\nOther line\n**Bold** e *italic*",
      "url": "https://oficial.com",
      "customFields": [
        {
          "label": "Other websites",
          "type": "url",
          "value": "https://other.oficial.com"
        },
        {
          "label": "Other wesite",
          "type": "url",
          "value": "https://another.oficial.com"
        }
      ],
      "folder": "00000000-0000-0000-0000-000000000000",
      "edited": 1774469678,
      "favorite": false,
      "tags": []
    },
    {
      "id": "d647383e-7373-4577-8d7b-2f12c016ac9a",
      "revision": "abed518a-ed50-4a30-ba41-d36b6f2050ba",
      "label": "The name",
      "username": "usuar2",
      "password": "password2",
      "notes": "more divagations",
      "url": "https:/site.com",
      "customFields": [],
      "folder": "00000000-0000-0000-0000-000000000000",
      "edited": 1774469781,
      "favorite": false,
      "tags": []
    }
  ],
  "folders": [],
  "tags": []
}
```

# Password Anaysis

Some code I use to analyze and try to optimise my own list of passwords.

Conclusions of this analysis makes me edit my labels and passwords in the iOS or macOS Passwords app, re-export, re-generate the JSON file and re-import it in Nextcloud Passwords.

In [ ]:
(
        pandas.read_csv(icloud_passwords)
        .rename(
            columns=dict(
                Title='label',
                URL='url',
                Username='username',
                Password='password',
                Notes='notes',
            )
        )

        # Make a hash for future aggregations
        .assign(
            final_hash=lambda table: table.apply(
                axis=1,
                func=lambda row: (
                    hashlib.shake_128(
                        '•'.join([
                            '' if pandas.isna(v) else str(v)
                            for v in [row.username,row.password,row.notes,row.OTPAuth]
                        ])
                        .encode()
                    )
                    .hexdigest(10)
                )
            ),
            file_time=lambda table: pandas.Timestamp.fromtimestamp(mtime - 5000),
            edited=lambda table: (table.file_time +  pandas.to_timedelta(table.index, unit="s")).astype(int) // 10**9,
            base_label=lambda table: table.label.str.extract(r'(.*) \(')
        )

        # Aggregate and define values for each field
        .groupby('final_hash')
        .agg(
            base_label=('base_label', 'first'),

            label=('label', 'first'),
            url=('url', 'first'),
            URLs=('url', list),
            username=('username', 'first'),
            password=('password', 'first'),
            notes=('notes', 'first'),
            edited=('edited','first'),
            OTPAuth=('OTPAuth', 'first'),
        )

        # [['base_label']]
        # .dropna()
        # .value_counts()
        # .tail(723-5)
        # .head(20)
)